# Model Serving with FastAPI

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 4/6

A model inside a notebook helps exactly one person. This lesson walks the model out of the notebook and behind a validated HTTP API with FastAPI — request schemas, automatic validation, health checks — and tests the whole thing with zero servers running, using FastAPI's own TestClient.

## 🎯 Learning Objectives

- Choose between batch scoring and a real-time API, and defend the choice
- Define routes with `@app.get` / `@app.post`, path and query parameters
- Declare typed request bodies with Pydantic models and let FastAPI reject bad input with 422s
- Load a trained pipeline ONCE at startup and expose `POST /predict` returning plain JSON types
- Drive the entire API from tests with `fastapi.testclient.TestClient` — no server, no ports
- Add `/health` readiness endpoints and outline how a container ships the service

## 1. From Notebook to Product

Two legitimate ways for applications to consume a model — they answer different questions:

| | Batch scoring | Real-time API |
|---|---|---|
| Question answered | "Score everyone overnight" | "Score THIS request right now" |
| Trigger | schedule (nightly cron/orchestrator) | user action / another service |
| Latency budget | minutes–hours OK | milliseconds–seconds |
| Typical home | warehouse tables, CSV drops | web/mobile app backend, microservice |
| Failure blast radius | tomorrow morning's report | the checkout page, right now |

Churn SMS campaigns are batch; fraud checks at card swipe are real-time. This lesson builds
the real-time shape — the discipline transfers directly to batch wrappers.

## 2. Hello, FastAPI

FastAPI is a modern Python web framework built on type hints: you declare *what* a route
accepts, and it handles parsing, validation and documentation. Routes are plain functions
wearing a decorator.

**Syntax:**

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/items/{item_id}")            # path parameter
def read_item(item_id: int, noisy: bool = False):   # query parameter with default
    return {"item_id": item_id, "noisy": noisy}
```

Run it (in a terminal, never needed for this lesson):

```bash
pip install "fastapi[standard]"
uvicorn main:app --reload               # serves http://127.0.0.1:8000
```

We will NOT start servers in this course. FastAPI ships a test client that talks to the
app in-process — same code paths, no sockets:

In [ ]:
# Routes + TestClient: a conversation without a server
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()


@app.get("/")
def root():
    return {"service": "score-api", "docs": "/docs"}


@app.get("/students/{name}")
def greet(name: str, age: int = 18):
    return {"message": f"Hello {name}", "age": age, "adult": age >= 18}


client = TestClient(app)

r = client.get("/")
print(r.status_code, r.json())

r = client.get("/students/Sarah", params={"age": 21})
print(r.status_code, r.json())

r = client.get("/students/Rahim", params={"age": "not-a-number"})   # int? really?
print(r.status_code, "<- FastAPI rejected the bad query automatically")

## 3. Typed Request Bodies: Pydantic Does the Bouncer Work

POST endpoints accept JSON bodies described by a **Pydantic model** — a class with typed,
constraint-carrying fields. If the payload violates the declaration, FastAPI refuses the
request with HTTP **422 Unprocessable Entity** before your function is ever called. Your
endpoint body only ever sees valid, correctly-typed data.

**Syntax:**

```python
from pydantic import BaseModel, Field

class StudentFeatures(BaseModel):
    hours_studied: float = Field(ge=0, le=168)   # per week; hard physical bounds
    attendance: float = Field(ge=0, le=1)
```

> 🔍 **Under the Hood:** Pydantic v2 validates in Rust-compiled cores and coerces
> aggressively-but-safely: `"3.5"` becomes `3.5`, `3` becomes `3.0`, but `"abc"` fails.
> Constraints like `ge`/`le` compile into the JSON schema that powers FastAPI's automatic
> `/docs` page — one declaration drives validation AND documentation AND editor hints.
> This is why the request model is the API's real contract, not the docstring.

In [ ]:
# The contract: valid payloads pass, garbage bounces with 422
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app = FastAPI()


class StudentFeatures(BaseModel):
    hours_studied: float = Field(ge=0, le=168)
    attendance: float = Field(ge=0, le=1)


@app.post("/echo")
def echo(features: StudentFeatures):
    return {"received": features.model_dump()}


client = TestClient(app)

good = client.post("/echo", json={"hours_studied": 12.5, "attendance": 0.92})
print("valid payload   ->", good.status_code, good.json())

bad = client.post("/echo", json={"hours_studied": 999, "attendance": 0.92})
print("hours_studied=999 ->", bad.status_code, "(out of range)")
print("first complaint :", bad.json()["detail"][0]["msg"])

missing = client.post("/echo", json={"hours_studied": 12.5})
print("missing field   ->", missing.status_code)

## 4. Serving the Trained Model

The serving pattern fits in four sentences. Train (or load) the pipeline **once, at module
level** — model loading is expensive and the weights never change mid-flight. Expose a
`POST /predict` whose request model mirrors the training-time feature schema. Call the
pipeline, convert NumPy scalars to built-in `float`/`int`/`bool` (JSON has no opinion about
`np.float32`), and return a dictionary describing the decision — including the model
version, so every response is auditable.

In [ ]:
# Study-hours model behind a validated API — trained once, served many times
import joblib
import numpy as np
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

MODEL_VERSION = "1.0.0"

# ---- pretend this came from the registry (Lesson 03) -----------------------
from sklearn.linear_model import LogisticRegression

HOURS_TRAIN = [[0.5], [1], [2], [3], [4], [5], [6], [7]]
PASS_TRAIN = [0, 0, 0, 1, 0, 1, 1, 1]
pipeline = LogisticRegression(C=1.0, max_iter=2000).fit(HOURS_TRAIN, PASS_TRAIN)


class StudentFeatures(BaseModel):
    hours_studied: float = Field(ge=0, le=80)


app = FastAPI(title="study-score-api")


@app.post("/predict")
def predict(features: StudentFeatures):
    proba = pipeline.predict_proba([[features.hours_studied]])[0][1]
    return {"hours_studied": features.hours_studied,
            "probability": round(float(proba), 4),       # np.float64 -> float
            "will_pass": bool(proba >= 0.5),             # np.bool_   -> bool
            "model_version": MODEL_VERSION}


client = TestClient(app)

r = client.post("/predict", json={"hours_studied": 3.5})
print(r.status_code, r.json())

r = client.post("/predict", json={"hours_studied": -1})   # nonsense hours
print(r.status_code, "<- rejected before our function ran")

## 5. Health Checks: the Endpoint That Watches the Watcher

Every service needs `GET /health`. Load balancers, Kubernetes probes and CI smoke tests all
ask it one question: *"are you alive and ready?"* Return the model version too — the first
question during any incident is "which model is even running?"

**Convention:** `200 {"status": "ok"}` when ready to serve; anything else means do-not-send-
traffic. Keep it dependency-free (don't call the model on every probe — just confirm it loaded).

In [ ]:
# /health + /version: tiny endpoints, outsized operational value
from fastapi import FastAPI
from fastapi.testclient import TestClient

MODEL_VERSION = "1.0.0"

app = FastAPI()


@app.get("/health")
def health():
    return {"status": "ok", "model_version": MODEL_VERSION}


client = TestClient(app)
r = client.get("/health")
print(r.status_code, r.json())
assert r.json()["status"] == "ok"
print("load balancers would route traffic here.")

## 6. Testing the Contract

An API is a promise: these inputs yield that response shape. Contract tests pin the promise.
Each test is a tiny function hitting the TestClient — happy path, boundary values, and the
rejections. Run them in CI (Lesson 06) and a breaking change dies in the pull request
instead of in production.

In [ ]:
# Contract tests for the study-score API — no server required
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

from sklearn.linear_model import LogisticRegression

pipeline = LogisticRegression(max_iter=2000).fit([[0.5], [1], [2], [3], [4], [5], [6], [7]],
                                                 [0, 0, 0, 1, 0, 1, 1, 1])
app = FastAPI()


class StudentFeatures(BaseModel):
    hours_studied: float = Field(ge=0, le=80)


@app.post("/predict")
def predict(features: StudentFeatures):
    proba = float(pipeline.predict_proba([[features.hours_studied]])[0][1])
    return {"probability": round(proba, 4), "will_pass": proba >= 0.5}


client = TestClient(app)


def test_predict_happy_path():
    body = client.post("/predict", json={"hours_studied": 3.5}).json()
    assert set(body) == {"probability", "will_pass"}
    assert 0.0 <= body["probability"] <= 1.0
    assert isinstance(body["will_pass"], bool)


def test_predict_rejects_impossible_hours():
    assert client.post("/predict", json={"hours_studied": -2}).status_code == 422
    assert client.post("/predict", json={"hours_studied": 500}).status_code == 422


tests_passed = 0
for test_fn in (test_predict_happy_path, test_predict_rejects_impossible_hours):
    test_fn()
    tests_passed += 1
print(f"contract tests passed: {tests_passed}/2")

## 7. Shipping It

A served model is a normal program plus weights: pin it, wrap it, ship it.

```text
# Dockerfile (sketch)
FROM python:3.12-slim
WORKDIR /srv
COPY requirements.lock .
RUN pip install --no-cache-dir -r requirements.lock
COPY model.joblib app.py .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

- `requirements.lock` pins fastapi, uvicorn, scikit-learn — the Lesson 02 habit, now load-bearing.
- The image contains code + weights together: the deployed artifact and the tested artifact
  become literally the same bytes.
- Behind it, Lesson 06's monitoring watches what this service sees.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Returning NumPy types (`np.float32`, `np.bool_`, arrays) | JSON encoder throws `Object of type float32 is not JSON serializable` | Cast at the boundary: `float(x)`, `bool(x)`, `.tolist()` |
| Loading the model inside the endpoint | Every request pays cold-start cost; latency spikes | Load once at module level; endpoints only infer |
| Hand-parsing the request body (`await request.json()` + hope) | Garbage reaches the model; 500s replace helpful 422s | Declare a Pydantic request model with bounds |
| Declaring `async def` then calling blocking sklearn/pandas | The event loop stalls; throughput collapses | Plain `def` endpoints run in a threadpool — right for inference |
| Testing by starting a real server | Flaky ports, slow suites, skipped tests | `TestClient` — same app, no sockets |
| No `/health` endpoint | Orchestrators restart a half-working service blindly | Return version + status; keep it cheap |
| Untyped feature dicts accepted at the border | Training-serving skew sneaks in one renamed key at a time | Mirror the training feature schema exactly in the request model |

## 💡 Best Practices & Pro Tips

- **Version the URL, not just the payload**: `/v1/predict` lets v2 evolve without holding
  clients hostage; deprecate loudly and slowly.
- **Return decisions, not internals**: `{"will_pass": true, "probability": 0.83}` beats
  leaking raw array outputs nobody can act on.
- **Echo the model version in every response** — incident triage starts there.
- **Log requests and predictions** (Lesson 01): today's prediction log is tomorrow's
  training data and this week's drift evidence.
- **Contract tests in CI** (Lesson 06): the suite from Section 6 belongs in the pull
  request, not in someone's muscle memory.
- **AI-engineering relevance:** LLM endpoints are this lesson with different cargo —
  validate prompts and temperatures with Pydantic, stream tokens instead of arrays,
  and watch token-cost-per-request the way you watch p95 latency.

## 📌 Summary

| Tool / Pattern | What it does | Example |
|---|---|---|
| `FastAPI()` + decorators | Declare routes as typed functions | `@app.post("/predict")` |
| Path / query parameters | Typed URL inputs with defaults | `def greet(name: str, age: int = 18)` |
| Pydantic `BaseModel` + `Field` | The request contract; auto 422s | `hours: float = Field(ge=0, le=80)` |
| Module-level pipeline | Load once, infer many times | `PIPELINE = joblib.load(...)` |
| `float()` / `bool()` / `.tolist()` | NumPy -> JSON at the boundary | `round(float(proba), 4)` |
| `GET /health` | Readiness + version for orchestrators | `{"status": "ok", "model_version": ...}` |
| `fastapi.testclient.TestClient` | Full API tests, zero servers | `client.post("/predict", json={...})` |

Key takeaways:

- Serving turns a model into a product surface; validation at the border is the product's immune system.
- Pydantic models are the contract: bad input dies with 422 before reaching your code.
- Load once, infer often, return built-in types, echo the model version.
- TestClient makes the whole API a unit-testable object — contract tests belong in CI.

## 🔗 Next Lesson

Up next: **[05_Testing_Data_Quality](../05_Testing_Data_Quality/notes.ipynb)** — the tests that guard what flows INTO the model: schemas, expectations, and behaviour contracts that block bad batches and bad models.